# Verwijsbrief Analyse — Orchestrator met State

De orchestrator beheert een centrale **`PipelineState`** die continu wordt bijgewerkt.
Elke tool leest en muteert deze state via `RunContext`. De orchestrator kan op elk
moment `bekijk_status` aanroepen om te zien wat compleet is en wat nog ontbreekt.

```
┌─────────────────────────────────────────────────────┐
│                   PipelineState                     │
│                                                     │
│  brief_tekst          ← invoer                      │
│  gegevens             ← na extractie                │
│  medische_analyse     ← na analyse                  │
│  dossier_resultaat    ← na dossiercheck             │
│  routering            ← na routeringsbeslissing     │
│  externe_contacten    ← na email-zoekactie          │
│  berichten            ← na berichtopstelling        │
│  antwoorden           ← bij ontvangen antwoorden    │
│  follow_ups           ← na follow-up analyse        │
│  overzicht            ← eindoverzicht               │
│                                                     │
│  informatie_items {}  ← per item: status tracker    │
│  voltooid_percentage  ← automatisch berekend        │
└─────────────────────────────────────────────────────┘
```

---
## Installatie

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from enum import Enum

import mlflow
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.capabilities import Thinking, WebSearch
from pydantic_ai.models.openai import OpenAIResponsesModelSettings

load_dotenv()

mlflow.set_tracking_uri("http://localhost:5068")
mlflow.set_experiment(f"agentic-ai-hackathon-{os.getenv('USER', 'unknown')}")
mlflow.pydantic_ai.autolog()

deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT", "")

settings = OpenAIResponsesModelSettings(
    openai_reasoning_effort="low",
    openai_reasoning_summary="detailed",
)

---
## Pipeline State

Eén muteerbaar object dat door alle tools gelezen en bijgewerkt wordt.
Elk informatie-item wordt getrackt met een status en bron.

In [ ]:
class ItemStatus(str, Enum):
    """Status van een individueel informatie-item."""
    ONTBREKEND = "ontbrekend"              # Niet in brief, niet in dossier
    IN_BRIEF = "in_brief"                  # Gevonden in de verwijsbrief
    IN_DOSSIER = "in_dossier"              # Gevonden in eigen dossier
    UITGEVRAAGD = "uitgevraagd"            # Bericht verstuurd, wacht op antwoord
    BEANTWOORD = "beantwoord"              # Antwoord ontvangen en verwerkt
    ONDUIDELIJK = "onduidelijk"            # Antwoord ontvangen maar onvolledig


@dataclass
class InformatieItem:
    """Eén stuk informatie dat getrackt wordt."""
    beschrijving: str
    aandoening: str
    categorie: str  # 'diagnose_oorzaak', 'diagnostiek', 'behandeling'
    status: ItemStatus = ItemStatus.ONTBREKEND
    waarde: str = ""
    bron: str = ""  # 'verwijsbrief', 'dossier', 'antwoord_verwijzer', 'antwoord_extern'
    ontvanger: str = ""  # 'verwijzer' of 'externe_behandelaar'


@dataclass
class PipelineState:
    """Centrale state die door de hele pipeline bijgewerkt wordt."""

    # ── Invoer ──
    brief_tekst: str = ""

    # ── Resultaten per stap (worden gevuld door tools) ──
    gegevens_json: str = ""                # Na extractie
    medische_analyse_json: str = ""        # Na medische analyse
    dossier_json: str = ""                 # Na dossiercheck
    routering_json: str = ""               # Na routering
    externe_contacten_json: str = ""       # Na email-zoekactie
    berichten_json: str = ""               # Na berichtopstelling
    overzicht_json: str = ""               # Eindoverzicht

    # ── Informatie tracker ──
    informatie_items: dict[str, InformatieItem] = field(default_factory=dict)

    # ── Conversatie-geschiedenis ──
    antwoorden: list[dict] = field(default_factory=list)  # [{afzender, tekst, datum}]
    follow_ups: list[str] = field(default_factory=list)    # Verstuurde vervolgberichten

    # ── Status ──
    stappen_uitgevoerd: list[str] = field(default_factory=list)

    def voeg_item_toe(self, key: str, beschrijving: str, aandoening: str,
                      categorie: str, status: ItemStatus = ItemStatus.ONTBREKEND,
                      waarde: str = "", bron: str = "") -> None:
        """Voeg een informatie-item toe of update het als het al bestaat."""
        self.informatie_items[key] = InformatieItem(
            beschrijving=beschrijving, aandoening=aandoening,
            categorie=categorie, status=status, waarde=waarde, bron=bron,
        )

    def update_item(self, key: str, status: ItemStatus,
                    waarde: str = "", bron: str = "") -> None:
        """Update de status van een bestaand item."""
        if key in self.informatie_items:
            self.informatie_items[key].status = status
            if waarde:
                self.informatie_items[key].waarde = waarde
            if bron:
                self.informatie_items[key].bron = bron

    @property
    def items_ontbrekend(self) -> list[InformatieItem]:
        return [i for i in self.informatie_items.values()
                if i.status == ItemStatus.ONTBREKEND]

    @property
    def items_uitgevraagd(self) -> list[InformatieItem]:
        return [i for i in self.informatie_items.values()
                if i.status == ItemStatus.UITGEVRAAGD]

    @property
    def items_compleet(self) -> list[InformatieItem]:
        return [i for i in self.informatie_items.values()
                if i.status in (ItemStatus.IN_BRIEF, ItemStatus.IN_DOSSIER, ItemStatus.BEANTWOORD)]

    @property
    def items_onduidelijk(self) -> list[InformatieItem]:
        return [i for i in self.informatie_items.values()
                if i.status == ItemStatus.ONDUIDELIJK]

    @property
    def voltooid_percentage(self) -> float:
        if not self.informatie_items:
            return 0.0
        return len(self.items_compleet) / len(self.informatie_items) * 100

    @property
    def alles_compleet(self) -> bool:
        return (
            len(self.informatie_items) > 0
            and len(self.items_ontbrekend) == 0
            and len(self.items_onduidelijk) == 0
            and len(self.items_uitgevraagd) == 0
        )

    def status_overzicht(self) -> str:
        """Geeft een leesbaar statusoverzicht als string."""
        lines = [
            f"=== STATUS OVERZICHT ===",
            f"Totaal items: {len(self.informatie_items)}",
            f"Compleet: {len(self.items_compleet)} | "
            f"Ontbrekend: {len(self.items_ontbrekend)} | "
            f"Uitgevraagd: {len(self.items_uitgevraagd)} | "
            f"Onduidelijk: {len(self.items_onduidelijk)}",
            f"Voltooid: {self.voltooid_percentage:.0f}%",
            f"Alles compleet: {self.alles_compleet}",
            f"Stappen uitgevoerd: {', '.join(self.stappen_uitgevoerd)}",
            "",
        ]
        for status_type in ItemStatus:
            items = [i for i in self.informatie_items.values() if i.status == status_type]
            if items:
                lines.append(f"--- {status_type.value.upper()} ---")
                for item in items:
                    bron_str = f" (bron: {item.bron})" if item.bron else ""
                    waarde_str = f" → {item.waarde}" if item.waarde else ""
                    lines.append(f"  [{item.aandoening}] {item.beschrijving}{waarde_str}{bron_str}")
                lines.append("")
        return "\n".join(lines)

---
## Pydantic output-modellen

In [ ]:
class VerwijsbriefGegevens(BaseModel):
    verwijsdatum_brief: str = Field(default="Unknown")
    bsn_patient: str = Field(default="Unknown")
    voorletters_patient: str = Field(default="Unknown")
    achternaam_patient: str = Field(default="Unknown")
    geboortedatum_patient: str = Field(default="Unknown")
    geslacht_patient: str = Field(default="Unknown")
    telefoonnummer_patient: str = Field(default="Unknown")
    mailadres_patient: str = Field(default="Unknown")
    adres_patient: str = Field(default="Unknown")
    naam_instantie: str = Field(default="Unknown")
    postcode_instantie: str = Field(default="Unknown")
    plaatsnaam_instantie: str = Field(default="Unknown")
    achternaam_verwijzer: str = Field(default="Unknown")
    agb_code_verwijzer: str = Field(default="Unknown")
    achternaam_huisarts: str = Field(default="Unknown")
    postcode_huisarts: str = Field(default="Unknown")
    plaatsnaam_huisarts: str = Field(default="Unknown")


class MedischeAandoening(BaseModel):
    aandoening: str
    diagnose_oorzaak: str = "Niet vermeld"
    diagnostiek: str = "Niet vermeld"
    behandeling: str = "Niet vermeld"
    extern_ziekenhuis: str = "Niet vermeld"
    externe_afdeling: str = "Niet vermeld"
    ontbrekende_informatie: list[str] = Field(default_factory=list)

class MedischeAnalyse(BaseModel):
    aandoeningen: list[MedischeAandoening]
    samenvatting_ontbrekend: list[str]


class DossierItem(BaseModel):
    ontbrekend_item: str
    gevonden_in_dossier: bool
    dossier_waarde: str = ""

class DossierCheckResultaat(BaseModel):
    gevonden_items: list[DossierItem]
    nog_ontbrekend: list[str]


class UitvraagItem(BaseModel):
    ontbrekend_item: str
    aandoening: str
    ontvanger: str  # 'verwijzer' of 'externe_behandelaar'
    reden: str
    extern_ziekenhuis: str = ""
    externe_afdeling: str = ""

class UitvraagRoutering(BaseModel):
    items_verwijzer: list[UitvraagItem]
    items_externe_behandelaar: list[UitvraagItem]


class ExternContactInfo(BaseModel):
    ziekenhuis: str
    afdeling: str
    email: str = "Niet gevonden"
    telefoon: str = "Niet gevonden"
    bron: str = ""

class ExternContactResultaat(BaseModel):
    contacten: list[ExternContactInfo]


class Bericht(BaseModel):
    ontvanger_type: str
    ontvanger_naam: str
    email_adres: str = ""
    onderwerp: str
    aanhef: str
    bericht_tekst: str
    afsluiting: str

class BerichtenPakket(BaseModel):
    berichten: list[Bericht]


class FollowUpAnalyse(BaseModel):
    beantwoorde_items: list[str]
    nog_onduidelijk: list[str]
    nieuwe_vragen: list[str]
    alles_compleet: bool
    vervolgbericht: str = ""


class AandoeningOverzicht(BaseModel):
    aandoening: str
    diagnose_oorzaak: str
    diagnostiek: str
    behandeling: str
    bron: str

class AssistentenOverzicht(BaseModel):
    patient_naam: str
    patient_geboortedatum: str
    patient_bsn: str
    verwijzer: str
    verwijsdatum: str
    aandoeningen: list[AandoeningOverzicht]
    samenvatting: str
    aandachtspunten: list[str]


class PipelineResultaat(BaseModel):
    redenering: str = Field(description="Uitleg welke stappen genomen zijn en waarom")
    status_samenvatting: str = Field(description="Huidige status: percentage compleet en wat nog openstaat")
    alles_compleet: bool = Field(description="True als alle informatie compleet is")

---
## Sub-agents

In [ ]:
extraction_agent = Agent(
    f"azure:{deployment}",
    instructions="Je extraheert gestructureerde gegevens uit Nederlandse verwijsbrieven. "
    "Alleen expliciete info. \"Unknown\" als iets ontbreekt. Datums: YYYY-MM-DD.",
    capabilities=[Thinking()], output_type=VerwijsbriefGegevens, model_settings=settings,
)

analyse_agent = Agent(
    f"azure:{deployment}",
    instructions="Je analyseert verwijsbrieven op medische volledigheid. Per aandoening: "
    "a) diagnose/oorzaak, b) diagnostiek, c) behandeling. Let op extern ziekenhuis/afdeling. "
    "Wees specifiek. Geen aannames. Nederlands.",
    capabilities=[Thinking()], output_type=MedischeAnalyse, model_settings=settings,
)

dossier_agent = Agent(
    f"azure:{deployment}",
    instructions="Je raadpleegt het eigen patiëntdossier via de tool 'zoek_in_dossier'. "
    "Vergelijk elk ontbrekend item met het dossier.",
    capabilities=[Thinking()], output_type=DossierCheckResultaat, model_settings=settings,
)

routing_agent = Agent(
    f"azure:{deployment}",
    instructions="Je bepaalt per ontbrekend item of de verwijzer of de externe behandelaar "
    "aangeschreven moet worden. Extern ziekenhuis → details naar dat ziekenhuis. Overige → verwijzer.",
    capabilities=[Thinking()], output_type=UitvraagRoutering, model_settings=settings,
)

email_agent = Agent(
    f"azure:{deployment}",
    instructions="Je zoekt contactgegevens van ziekenhuisafdelingen via websearch. "
    "Zoek e-mail en telefoon van het secretariaat. Gebruik officiële bronnen. Geef bron-URL mee.",
    capabilities=[Thinking(), WebSearch(local="duckduckgo")],
    output_type=ExternContactResultaat, model_settings=settings,
)

bericht_agent = Agent(
    f"azure:{deployment}",
    instructions="Je stelt professionele berichten op in het Nederlands. Per ontvanger apart. "
    "Noem patiënt bij naam en geboortedatum. Wees specifiek. Formeel maar collegiaal.",
    capabilities=[Thinking()], output_type=BerichtenPakket, model_settings=settings,
)

followup_agent = Agent(
    f"azure:{deployment}",
    instructions="Je beoordeelt antwoorden op uitvragen. Wat is beantwoord? Wat is onduidelijk? "
    "Nieuwe vragen? Stel vervolgbericht op als nodig. Nederlands.",
    capabilities=[Thinking()], output_type=FollowUpAnalyse, model_settings=settings,
)

overzicht_agent = Agent(
    f"azure:{deployment}",
    instructions="Je maakt een eindoverzicht voor de doktersassistent. Combineer alle bronnen. "
    "Per aandoening: oorzaak, diagnostiek, behandeling, bron. Samenvatting 2-3 zinnen. Nederlands.",
    capabilities=[Thinking()], output_type=AssistentenOverzicht, model_settings=settings,
)

In [ ]:
# Mock dossier (vervang met EPD/HIS-koppeling)
MOCK_DOSSIER = {
    "123456789": {
        "bekende_aandoeningen": [
            {"aandoening": "Hypertensie", "sinds": "2023", "medicatie": "Lisinopril 10mg 1dd1"}
        ],
        "laboratorium": [
            {"datum": "2024-11-01", "test": "Totaal cholesterol", "uitslag": "6.8 mmol/L (verhoogd)"}
        ],
        "notities": "Patiënte is in 2023 gezien door cardiologie UMC Utrecht i.v.m. "
        "verdenking secundaire hypertensie. Geen verdere gegevens ontvangen."
    }
}

@dossier_agent.tool_plain
def zoek_in_dossier(bsn: str) -> str:
    """Zoek patiëntgegevens in het eigen dossier aan de hand van BSN.

    Args:
        bsn: BSN van de patiënt (9 cijfers).
    Returns:
        JSON met dossierinhoud.
    """
    dossier = MOCK_DOSSIER.get(bsn)
    if dossier:
        return json.dumps(dossier, ensure_ascii=False, indent=2)
    return json.dumps({"melding": f"Geen dossier gevonden voor BSN {bsn}"})

---
## Orchestrator met `deps_type=PipelineState`

Elke tool ontvangt `ctx: RunContext[PipelineState]` en werkt `ctx.deps` bij.

In [ ]:
orchestrator = Agent(
    f"azure:{deployment}",
    deps_type=PipelineState,
    instructions="""
Je bent de orchestrator van een verwijsbrief-analysepipeline. Je hebt toegang tot
een centrale state die je continu kunt raadplegen via 'bekijk_status'.

== WERKWIJZE ==

1. ALTIJD uitvoeren:
   - extraheer_gegevens
   - analyseer_medische_info
   - check_dossier

2. Na elke stap: roep 'bekijk_status' aan om te zien wat de huidige stand is.

3. BESLISSEN op basis van de status:
   - Zijn er nog items met status 'ontbrekend'?
     → JA: bepaal_routering, dan eventueel zoek_extern_contact, dan stel_berichten_op
     → NEE: ga direct naar het overzicht
   - Zijn er items naar een extern ziekenhuis?
     → JA: zoek_extern_contact
     → NEE: sla over

4. ALTIJD als laatste: maak_overzicht

== BIJ FOLLOW-UP ==

Als je een antwoord ontvangt:
- verwerk_antwoord → bekijk_status
- Nog onduidelijk? → stel vervolgbericht op
- Alles compleet? → bijgewerkt overzicht maken

== REGELS ==

- Gebruik 'bekijk_status' als je kompas — het percentage en de itemstatussen
  vertellen je precies waar je staat.
- Beschrijf in je 'redenering' welke beslissingen je hebt genomen en waarom.
- Reageer in het Nederlands.
""",
    capabilities=[Thinking()],
    output_type=PipelineResultaat,
    model_settings=settings,
)

In [ ]:
# ── Status tool ──────────────────────────────────────────────────────────

@orchestrator.tool
async def bekijk_status(ctx: RunContext[PipelineState]) -> str:
    """Bekijk de huidige status van de pipeline: welke items zijn compleet,
    welke ontbreken nog, en wat is het voltooiingspercentage.

    Returns:
        Leesbaar statusoverzicht.
    """
    return ctx.deps.status_overzicht()

In [ ]:
# ── Extractie tool ───────────────────────────────────────────────────────

@orchestrator.tool
async def extraheer_gegevens(ctx: RunContext[PipelineState], brief_tekst: str) -> str:
    """Extraheer administratieve en patiëntgegevens uit de verwijsbrief.

    Args:
        brief_tekst: De volledige tekst van de verwijsbrief.
    Returns:
        JSON met geëxtraheerde gegevens.
    """
    result = await extraction_agent.run(
        f"Extraheer alle velden uit de volgende verwijsbrief:\n\n{brief_tekst}"
    )
    ctx.deps.gegevens_json = result.output.model_dump_json(indent=2)
    ctx.deps.brief_tekst = brief_tekst
    ctx.deps.stappen_uitgevoerd.append("extractie")
    return ctx.deps.gegevens_json

In [ ]:
# ── Medische analyse tool ────────────────────────────────────────────────

@orchestrator.tool
async def analyseer_medische_info(ctx: RunContext[PipelineState], brief_tekst: str) -> str:
    """Analyseer de verwijsbrief op medische volledigheid per aandoening.

    Args:
        brief_tekst: De volledige tekst van de verwijsbrief.
    Returns:
        JSON met medische analyse. Vult ook de informatie-items in de state.
    """
    result = await analyse_agent.run(
        f"Analyseer de volgende verwijsbrief op medische volledigheid:\n\n{brief_tekst}"
    )
    analyse = result.output
    ctx.deps.medische_analyse_json = analyse.model_dump_json(indent=2)

    # Vul de informatie-items in de state
    for aand in analyse.aandoeningen:
        for cat, waarde in [
            ("diagnose_oorzaak", aand.diagnose_oorzaak),
            ("diagnostiek", aand.diagnostiek),
            ("behandeling", aand.behandeling),
        ]:
            key = f"{aand.aandoening}::{cat}"
            if waarde and waarde != "Niet vermeld":
                ctx.deps.voeg_item_toe(
                    key, f"{cat} van {aand.aandoening}", aand.aandoening,
                    cat, ItemStatus.IN_BRIEF, waarde, "verwijsbrief",
                )
            else:
                ctx.deps.voeg_item_toe(
                    key, f"{cat} van {aand.aandoening}", aand.aandoening, cat,
                )

    ctx.deps.stappen_uitgevoerd.append("medische_analyse")
    return ctx.deps.medische_analyse_json

In [ ]:
# ── Dossier check tool ───────────────────────────────────────────────────

@orchestrator.tool
async def check_dossier(ctx: RunContext[PipelineState], bsn: str, ontbrekende_items: list[str]) -> str:
    """Controleer of ontbrekende informatie al in het eigen dossier staat.

    Args:
        bsn: BSN van de patiënt.
        ontbrekende_items: Lijst van ontbrekende items.
    Returns:
        JSON met dossiercheck-resultaat. Update de state voor gevonden items.
    """
    items_tekst = "\n".join(f"- {item}" for item in ontbrekende_items)
    result = await dossier_agent.run(
        f"BSN: {bsn}\n\nOntbrekende items:\n{items_tekst}"
    )
    dossier = result.output
    ctx.deps.dossier_json = dossier.model_dump_json(indent=2)

    # Update state: gevonden items markeren
    for item in dossier.gevonden_items:
        if item.gevonden_in_dossier:
            # Zoek matching key in state
            for key, info in ctx.deps.informatie_items.items():
                if info.status == ItemStatus.ONTBREKEND and (
                    item.ontbrekend_item.lower() in info.beschrijving.lower()
                    or info.beschrijving.lower() in item.ontbrekend_item.lower()
                ):
                    ctx.deps.update_item(
                        key, ItemStatus.IN_DOSSIER, item.dossier_waarde, "eigen dossier"
                    )

    ctx.deps.stappen_uitgevoerd.append("dossier_check")
    return ctx.deps.dossier_json

In [ ]:
# ── Routering tool ───────────────────────────────────────────────────────

@orchestrator.tool
async def bepaal_routering(ctx: RunContext[PipelineState], ontbrekende_items: list[str]) -> str:
    """Bepaal per ontbrekend item of verwijzer of externe behandelaar aangeschreven moet worden.

    Args:
        ontbrekende_items: Items die nog uitgevraagd moeten worden.
    Returns:
        JSON met routeringsbeslissing. Update de state met ontvanger per item.
    """
    items_tekst = "\n".join(f"- {item}" for item in ontbrekende_items)
    result = await routing_agent.run(
        f"Ontbrekende items:\n{items_tekst}\n\n"
        f"Medische analyse:\n{ctx.deps.medische_analyse_json}"
    )
    routering = result.output
    ctx.deps.routering_json = routering.model_dump_json(indent=2)

    # Update state met ontvanger info
    for item in routering.items_verwijzer + routering.items_externe_behandelaar:
        for key, info in ctx.deps.informatie_items.items():
            if info.status == ItemStatus.ONTBREKEND and (
                item.ontbrekend_item.lower() in info.beschrijving.lower()
                or info.beschrijving.lower() in item.ontbrekend_item.lower()
            ):
                info.ontvanger = item.ontvanger

    ctx.deps.stappen_uitgevoerd.append("routering")
    return ctx.deps.routering_json

In [ ]:
# ── Email zoek tool ──────────────────────────────────────────────────────

@orchestrator.tool
async def zoek_extern_contact(ctx: RunContext[PipelineState], ziekenhuizen_afdelingen: list[str]) -> str:
    """Zoek e-mailadressen van secretariaten van externe ziekenhuisafdelingen.

    Args:
        ziekenhuizen_afdelingen: Lijst van 'Ziekenhuis - Afdeling' strings.
    Returns:
        JSON met contactgegevens.
    """
    zoek_tekst = "\n".join(f"- {z}" for z in ziekenhuizen_afdelingen)
    result = await email_agent.run(
        f"Zoek contactgegevens van het secretariaat van:\n{zoek_tekst}"
    )
    ctx.deps.externe_contacten_json = result.output.model_dump_json(indent=2)
    ctx.deps.stappen_uitgevoerd.append("extern_contact")
    return ctx.deps.externe_contacten_json

In [ ]:
# ── Bericht tool ─────────────────────────────────────────────────────────

@orchestrator.tool
async def stel_berichten_op(ctx: RunContext[PipelineState]) -> str:
    """Stel berichten op aan verwijzer en/of externe behandelaars.
    Gebruikt de gegevens, routering en contactinfo uit de state.

    Returns:
        JSON met opgestelde berichten. Markeert items als 'uitgevraagd' in de state.
    """
    prompt = (
        f"Stel berichten op.\n\n"
        f"PATIËNTGEGEVENS:\n{ctx.deps.gegevens_json}\n\n"
        f"ROUTERING:\n{ctx.deps.routering_json}\n\n"
    )
    if ctx.deps.externe_contacten_json:
        prompt += f"CONTACTGEGEVENS EXTERN:\n{ctx.deps.externe_contacten_json}\n"

    result = await bericht_agent.run(prompt)
    ctx.deps.berichten_json = result.output.model_dump_json(indent=2)

    # Markeer alle ontbrekende items als uitgevraagd
    for key, info in ctx.deps.informatie_items.items():
        if info.status == ItemStatus.ONTBREKEND:
            info.status = ItemStatus.UITGEVRAAGD

    ctx.deps.stappen_uitgevoerd.append("berichten")
    return ctx.deps.berichten_json

In [ ]:
# ── Follow-up tool ───────────────────────────────────────────────────────

@orchestrator.tool
async def verwerk_antwoord(
    ctx: RunContext[PipelineState],
    oorspronkelijk_bericht: str,
    ontvangen_antwoord: str,
    uitgevraagde_items: list[str],
) -> str:
    """Verwerk een ontvangen antwoord en update de state.

    Args:
        oorspronkelijk_bericht: Het eerder verstuurde bericht.
        ontvangen_antwoord: Het ontvangen antwoord.
        uitgevraagde_items: De items die uitgevraagd waren.
    Returns:
        JSON met follow-up analyse.
    """
    items_tekst = "\n".join(f"- {i}" for i in uitgevraagde_items)
    result = await followup_agent.run(
        f"OORSPRONKELIJK BERICHT:\n{oorspronkelijk_bericht}\n\n"
        f"ANTWOORD:\n{ontvangen_antwoord}\n\n"
        f"UITGEVRAAGDE ITEMS:\n{items_tekst}"
    )
    followup = result.output

    # Update state op basis van follow-up
    for beantwoord in followup.beantwoorde_items:
        for key, info in ctx.deps.informatie_items.items():
            if info.status == ItemStatus.UITGEVRAAGD and (
                beantwoord.lower() in info.beschrijving.lower()
                or info.beschrijving.lower() in beantwoord.lower()
            ):
                ctx.deps.update_item(key, ItemStatus.BEANTWOORD, bron="antwoord")

    for onduidelijk in followup.nog_onduidelijk:
        for key, info in ctx.deps.informatie_items.items():
            if info.status == ItemStatus.UITGEVRAAGD and (
                onduidelijk.lower() in info.beschrijving.lower()
                or info.beschrijving.lower() in onduidelijk.lower()
            ):
                ctx.deps.update_item(key, ItemStatus.ONDUIDELIJK)

    # Bewaar antwoord in geschiedenis
    ctx.deps.antwoorden.append({"tekst": ontvangen_antwoord})
    if followup.vervolgbericht:
        ctx.deps.follow_ups.append(followup.vervolgbericht)

    ctx.deps.stappen_uitgevoerd.append("follow_up")
    return followup.model_dump_json(indent=2)

In [ ]:
# ── Overzicht tool ───────────────────────────────────────────────────────

@orchestrator.tool
async def maak_overzicht(ctx: RunContext[PipelineState]) -> str:
    """Maak het eindoverzicht voor de doktersassistent uit alle verzamelde info in de state.

    Returns:
        JSON met het overzicht.
    """
    # Bouw context op uit de state
    alle_info = f"VERWIJSBRIEF:\n{ctx.deps.brief_tekst}\n\n"
    alle_info += f"GEGEVENS:\n{ctx.deps.gegevens_json}\n\n"
    if ctx.deps.dossier_json:
        alle_info += f"DOSSIER:\n{ctx.deps.dossier_json}\n\n"
    for antw in ctx.deps.antwoorden:
        alle_info += f"ONTVANGEN ANTWOORD:\n{antw['tekst']}\n\n"
    alle_info += f"STATUS:\n{ctx.deps.status_overzicht()}"

    result = await overzicht_agent.run(
        f"Maak een volledig overzicht voor de doktersassistent.\n\n{alle_info}"
    )
    ctx.deps.overzicht_json = result.output.model_dump_json(indent=2)
    ctx.deps.stappen_uitgevoerd.append("overzicht")
    return ctx.deps.overzicht_json

---
## Voorbeeldbrief

In [ ]:
letter_text = """
Huisartsenpraktijk De Linden
Lindenlaan 12
1234 AB Amsterdam
AGB-code: 01234567

Datum: 15-03-2025

Betreft: Verwijzing naar polikliniek Interne Geneeskunde

Geachte collega,

Hierbij verwijs ik bovengenoemde patiënt naar uw polikliniek.

Patiëntgegevens:
Naam: J.A. de Vries
BSN: 123456789
Geboortedatum: 20-05-1990
Geslacht: Vrouw
Telefoon: 06-12345678
E-mail: j.devries@email.nl
Adres: Kerkstraat 45, 1234 CD Amsterdam

Reden van verwijzing:
Patiënte is sinds 2023 bekend met hypertensie. Zij is hiervoor behandeld
bij de afdeling Cardiologie van het UMC Utrecht.
Daarnaast is bij routine bloedonderzoek een verhoogd cholesterol vastgesteld.
Patiënte klaagt ook over toenemende vermoeidheid sinds enkele maanden.

Graag uw beoordeling en eventueel aanvullend onderzoek.

Met vriendelijke groet,
Dr. B. Jansen
Huisarts
"""

---
## Pipeline starten

Maak de state aan en geef hem mee als dependency.

In [ ]:
# Maak een verse state aan
state = PipelineState(brief_tekst=letter_text)

# Start de orchestrator — hij krijgt de state mee als dependency
pipeline_result = await orchestrator.run(
    f"Verwerk de volgende verwijsbrief volledig:\n\n{letter_text}",
    deps=state,
)

print(pipeline_result.output.redenering)

In [ ]:
# Bekijk de live state — dit is hetzelfde object dat de orchestrator bijwerkt
print(state.status_overzicht())

In [ ]:
# Overzicht uit de state
if state.overzicht_json:
    ov = AssistentenOverzicht.model_validate_json(state.overzicht_json)
    print(f"Patiënt:       {ov.patient_naam}")
    print(f"Geboortedatum: {ov.patient_geboortedatum}")
    print(f"Verwijzer:     {ov.verwijzer}")
    print(f"\nSamenvatting:\n{ov.samenvatting}")
    for a in ov.aandoeningen:
        print(f"\n  ▸ {a.aandoening}")
        print(f"    Oorzaak:     {a.diagnose_oorzaak}")
        print(f"    Diagnostiek: {a.diagnostiek}")
        print(f"    Behandeling: {a.behandeling}")
        print(f"    Bron:        {a.bron}")
    for i, p in enumerate(ov.aandachtspunten, 1):
        print(f"  {i}. {p}")

In [ ]:
# Berichten uit de state
if state.berichten_json:
    berichten = BerichtenPakket.model_validate_json(state.berichten_json)
    for b in berichten.berichten:
        print(f"\n{'='*60}")
        print(f"AAN: {b.ontvanger_naam} ({b.ontvanger_type})")
        if b.email_adres:
            print(f"EMAIL: {b.email_adres}")
        print(f"ONDERWERP: {b.onderwerp}\n{'='*60}")
        print(f"{b.aanhef}\n\n{b.bericht_tekst}\n\n{b.afsluiting}\n[uw naam]")
else:
    print("Geen berichten nodig — alle info was al compleet.")

---
## Follow-up: antwoord verwerken

Geef de **dezelfde `state`** mee — de orchestrator ziet dan de volledige
geschiedenis en kan de status bijwerken.

In [ ]:
ontvangen_antwoord = """
Geachte collega,

In antwoord op uw verzoek over mevrouw De Vries:

Het verhoogde cholesterol betreft een totaal cholesterol van 6.8 mmol/L,
vastgesteld bij routine bloedonderzoek op 01-11-2024. Er is nog geen
medicamenteuze behandeling gestart, wel leefstijladvies gegeven.

De vermoeidheid is niet nader onderzocht.

Met vriendelijke groet,
Dr. B. Jansen
"""

antwoord_afzender = "verwijzer"

In [ ]:
# Zelfde state, nieuwe orchestrator-aanroep
followup_result = await orchestrator.run(
    f"Er is een antwoord binnengekomen van de {antwoord_afzender}.\n\n"
    f"ANTWOORD:\n{ontvangen_antwoord}\n\n"
    f"Verwerk dit antwoord, update de status, en bepaal vervolgstappen.",
    deps=state,  # ← zelfde state-object!
)

print(followup_result.output.redenering)
print(f"\nAlles compleet: {followup_result.output.alles_compleet}")

In [ ]:
# State is bijgewerkt — bekijk de nieuwe status
print(state.status_overzicht())

Herhaal de follow-up cellen voor elk nieuw antwoord. De `state` groeit mee.

---
## Debug

In [ ]:
# Alle tool calls van de orchestrator
# pipeline_result.all_messages()

# Ruwe state als dict
# import dataclasses
# dataclasses.asdict(state)